<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p>
    <p>Semana 05: No supervisado y segmentación</p>
  </div>
</div>

# 05 · No supervisado y segmentación

Hasta ahora teníamos una **respuesta** que predecir (el precio, la fuga). Hoy **no hay etiqueta**:
en vez de predecir, vamos a **descubrir estructura** en los datos.

Haremos tres cosas, todas *no supervisadas*:

1. **Segmentar** clientes con **k-means** (agrupar a los parecidos).
2. **Reducir dimensiones** con **PCA** para *ver* muchas variables en pocos ejes.
3. **Detectar anomalías** con **Isolation Forest** (lo que se sale del patrón).

> Regla de oro de hoy: sin etiqueta **no hay *accuracy***. El árbitro final es la **utilidad de negocio**.

## Preparación del entorno

Instalamos las librerías. `pyod` es el motor de detección de anomalías de PyCaret y `kneed`
detecta el "codo" automáticamente. La salida queda oculta.

In [1]:
%%capture
!pip install -q pycaret==4.0.0a8 plotly ipywidgets pyod kneed

## Setup técnico

Importaciones y configuración. No es contenido de la clase; deja el entorno listo.

<!-- migracion-modular: api-nuevas -->
## APIs de pandas que se incorporan

- `Series` representa una sola columna de una tabla; `set_option` ajusta cómo pandas muestra resultados sin cambiar los datos.


In [2]:
%matplotlib inline
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn import set_config
set_config(display='text')   # evita el pipeline gigante como HTML
pd.set_option('display.max_columns', 30)

## 1. Los datos: los mismos clientes, sin la etiqueta

Reutilizamos `clientes.csv` de la Clase 4 (~2.000 clientes de telecom), pero **quitamos `fuga`**:
hoy no hay respuesta que predecir.

In [3]:
REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO

df = pd.read_csv(BASE + 'clientes.csv').drop(columns='fuga')
print('Filas y columnas:', df.shape)
df.head()

Filas y columnas: (2000, 5)


,edad,antiguedad_meses,plan,cargo_mensual,llamadas_soporte
0,19,36,Estándar,55,1
1,73,1,Básico,34,6
2,42,11,Básico,16,3
3,31,35,Básico,25,1
4,70,5,Básico,33,4


### ¿Con qué variables agrupamos?

Segmentaremos por **comportamiento**: `antiguedad_meses`, `cargo_mensual` y `llamadas_soporte`.
Dejamos fuera `edad` a propósito: agrupar por una variable **sensible** puede llevar a decisiones
injustas (sesgo). Todas son numéricas → el codo y el silhouette se calculan sin problemas.

In [4]:
variables = ['antiguedad_meses', 'cargo_mensual', 'llamadas_soporte']
clientes = df[variables]
clientes.head()

,antiguedad_meses,cargo_mensual,llamadas_soporte
0,36,55,1
1,1,34,6
2,11,16,3
3,35,25,1
4,5,33,4


## 2. Segmentar con k-means (PyCaret)

`ClusteringExperiment` prepara los datos (con `normalize=True` **estandariza** — obligatorio, porque
k-means decide por **distancias**). Fíjate en lo que **falta**: no hay `target=` ni `compare_models`.
Sin etiqueta no hay ranking de modelos: elegir y validar pasa a ser **nuestro** trabajo.

In [5]:
from pycaret.tasks import ClusteringExperiment

exp = ClusteringExperiment(session_id=42, normalize=True).fit(clientes)
res = exp.create_model('kmeans', n_clusters=3)     # k=3 de partida; lo justificamos abajo
segmentos = exp.assign_model(res.pipeline)           # agrega la columna 'Cluster'
segmentos.head()

,antiguedad_meses,cargo_mensual,llamadas_soporte,Cluster
0,36,55,1,Cluster 1
1,1,34,6,Cluster 0
2,11,16,3,Cluster 0
3,35,25,1,Cluster 1
4,5,33,4,Cluster 0


## 3. ¿Cuántos grupos (k)? El codo y el silhouette

k-means **no** decide k por nosotros. Dos señales ayudan: el **codo** (la inercia baja al agregar
grupos; buscamos el **quiebre**) y el **silhouette** (qué tan separado queda cada punto, de -1 a 1;
buscamos el **máximo**).

### La vía rápida de PyCaret

PyCaret trae el gráfico del codo listo y marca el quiebre solo:

In [6]:
exp.plot_model(res.pipeline, plot='elbow')

**Ojo con la escala.** Ese gráfico rápido se calcula sobre los datos **sin estandarizar**, así que el
codo lo domina la variable de mayor escala (aquí `cargo_mensual`, en miles). Pero el clustering **sí**
estandariza. Hagámoslo **bien**: calculamos el codo y el silhouette sobre los datos **escalados** —
y dejamos que `kneed` sugiera el codo (es una sugerencia, no un veredicto).

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from kneed import KneeLocator

X = StandardScaler().fit_transform(clientes)
Ks = range(1, 9)
inercia, silueta = [], []
for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inercia.append(km.inertia_)
    silueta.append(silhouette_score(X, km.labels_) if k > 1 else np.nan)

codo = KneeLocator(list(Ks), inercia, curve='convex', direction='decreasing').elbow
print('Codo sugerido por kneed:', codo)
print('Silhouette máximo en k =', int(np.nanargmax(silueta)) + 1)

Codo sugerido por kneed: 4
Silhouette máximo en k = 2


In [8]:
fig = px.line(x=list(Ks), y=inercia, markers=True,
              labels={'x': 'Número de grupos (k)', 'y': 'Inercia (dispersión interna)'},
              title='Método del codo')
if codo:
    fig.add_vline(x=codo, line_dash='dash', line_color='gray')
fig.show()

In [9]:
fig = px.line(x=list(Ks)[1:], y=silueta[1:], markers=True,
              labels={'x': 'Número de grupos (k)', 'y': 'Silhouette'},
              title='Silhouette según k')
fig.show()

**Compara los dos codos.** El "rápido" de PyCaret (sin escalar) y este (escalado) pueden **diferir**:
moraleja, **escalar importa** también para elegir k. Y **ojo con datos reales**: en `clientes.csv` la
curva escalada es bastante **plana** (estructura de grupos débil) — eso también es información: aquí
**el negocio decide** cuántos segmentos puede gestionar. Con datos de estructura clara, el codo y el
máximo se ven mucho más marcados.

## 4. Perfilar y nombrar los segmentos

El algoritmo entrega grupos **sin nombre** ("Cluster 0, 1, 2"). Para decidir, miramos el **promedio
de cada variable por grupo** y traducimos a lenguaje de negocio.

In [10]:
perfil = segmentos.groupby('Cluster')[variables].mean().round(1)
perfil['n_clientes'] = segmentos.groupby('Cluster').size()
perfil

,antiguedad_meses,cargo_mensual,llamadas_soporte,n_clientes
Cluster,,,,
Cluster 0,11.2,48.3,2.9,461
Cluster 1,45.7,37.4,0.9,992
Cluster 2,42.2,79.5,1.2,547


Con la tabla de promedios traducimos cada grupo a **lenguaje de negocio**. Guía (pauta, no única
respuesta): *Leales de alto valor* (cargo alto, antigüedad alta, pocas llamadas) → beneficios y venta
cruzada · *Nuevos en riesgo* (antigüedad baja, muchas llamadas) → acompañamiento temprano · *Básicos
estables* (cargo bajo, sin problemas) → mantener a costo mínimo.

### Practiquemos juntos: ¿y si probamos k = 4?

Antes de correrlo, **anticipa**: ¿4 grupos darán un perfil accionable **nuevo**, o solo **partirán**
uno de los 3 en dos? Cambiemos `n_clusters` y comparemos los promedios.

In [11]:
res4 = exp.create_model('kmeans', n_clusters=4)
seg4 = exp.assign_model(res4.pipeline)
seg4.groupby('Cluster')[variables].mean().round(1)

,antiguedad_meses,cargo_mensual,llamadas_soporte
Cluster,,,
Cluster 0,10.2,53.3,3.8
Cluster 1,21.9,41.8,1.0
Cluster 2,42.2,84.6,1.3
Cluster 3,57.5,39.9,1.0


Compara con la tabla de 3 grupos: ¿el cuarto grupo aporta un segmento con **acción propia**, o es
uno de los anteriores dividido? Recuerda: **3 segmentos accionables valen más que 15** que nadie
puede gestionar. La métrica orienta; el **negocio decide**.

## 5. Reducir dimensiones con PCA

**PCA** (análisis de componentes principales) es una **técnica de reducción de dimensionalidad**:
resume muchas columnas en pocos **ejes nuevos** (componentes) conservando la mayor **variación**
posible. **No predice ni clasifica** — solo resume.

Un caso con **muchas** variables: tumores de mama, con **30 mediciones** por tumor (dataset clásico,
incluido en scikit-learn). Tenemos el diagnóstico (benigno/maligno), pero **PCA no lo usará**.

In [12]:
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA

cancer = load_breast_cancer(as_frame=True)
X_c = StandardScaler().fit_transform(cancer.data)          # estandarizar: obligatorio en PCA
diag = cancer.target.map({0: 'maligno', 1: 'benigno'})
print('Tumores x variables:', cancer.data.shape)
cancer.data.head()

Tumores x variables: (569, 30)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [13]:
pca = PCA(n_components=2).fit(X_c)
Z = pca.transform(X_c)
pve = pca.explained_variance_ratio_

# Sin colorear: la estructura ya aparece sola
fig = px.scatter(x=Z[:, 0], y=Z[:, 1], opacity=0.6,
                 labels={'x': f'Componente 1 ({pve[0]:.0%})', 'y': f'Componente 2 ({pve[1]:.0%})'},
                 title='30 variables comprimidas a 2 ejes (sin etiqueta)')
fig.show()

In [14]:
# Coloreado por diagnóstico (que PCA nunca vio): las zonas SON benigno/maligno
fig = px.scatter(x=Z[:, 0], y=Z[:, 1], color=diag, opacity=0.6,
                 color_discrete_map={'benigno': '#4C78A8', 'maligno': '#E45756'},
                 labels={'x': f'Componente 1 ({pve[0]:.0%})', 'y': f'Componente 2 ({pve[1]:.0%})', 'color': 'diagnóstico'},
                 title='Las mismas posiciones, coloreadas por diagnóstico')
fig.show()

PCA **no clasificó** nada: los tumores parecidos comparten mediciones y quedan cerca. La estructura ya estaba en los datos; PCA solo la dejó **a la vista**. Y en **3D** se ve aún mejor:

In [15]:
Z3 = PCA(n_components=3).fit_transform(X_c)
fig = px.scatter_3d(x=Z3[:, 0], y=Z3[:, 1], z=Z3[:, 2], color=diag,
                    color_discrete_map={'benigno': '#4C78A8', 'maligno': '#E45756'},
                    labels={'x': 'C1', 'y': 'C2', 'z': 'C3', 'color': 'diagnóstico'},
                    title='PCA en 3D (rotable con el mouse)')
fig.update_traces(marker_size=3)
fig.show()

¿Qué **significa** cada eje? Los **loadings** dicen cuánto pesa cada variable en el Componente 1:

In [16]:
loadings = pd.Series(pca.components_[0], index=cancer.data.columns)
top = loadings.reindex(loadings.abs().sort_values(ascending=False).index).head(8)
fig = px.bar(x=top.values[::-1], y=top.index[::-1], orientation='h',
             labels={'x': 'Peso en el Componente 1', 'y': ''},
             title='Loadings del Componente 1 (≈ tamaño y forma del tumor)')
fig.show()

## 6. Detección de anomalías: Isolation Forest

Volvamos a los clientes. **Isolation Forest** busca lo que **se sale del patrón** haciendo cortes al
azar: lo raro se **aísla** con pocos cortes. `AnomalyExperiment` lo corre en PyCaret; `fraction`
fija la proporción esperada de raros (aquí 5%).

In [17]:
from pycaret.tasks import AnomalyExperiment

ax = AnomalyExperiment(session_id=42).fit(clientes)
iso = ax.create_model('iforest', fraction=0.05)     # 5% esperado de anomalías
raros = ax.assign_model(iso.pipeline)                # + columnas 'Anomaly' (1/0) y 'Anomaly_Score'
print('Clientes marcados como anómalos:', int(raros['Anomaly'].sum()), 'de', len(raros))
raros.sort_values('Anomaly_Score', ascending=False).head(8)

Clientes marcados como anómalos: 100 de 2000


,antiguedad_meses,cargo_mensual,llamadas_soporte,Anomaly,Anomaly_Score
1129,5,111,6,1,0.113968
795,32,91,8,1,0.107785
1355,1,124,5,1,0.103663
353,11,100,8,1,0.101607
866,71,116,3,1,0.100291
375,36,116,5,1,0.097854
1934,10,107,6,1,0.094038
45,71,92,5,1,0.093963


El puntaje indica **rareza, no culpa**. Una anomalía puede ser un fraude… o tu mejor cliente. Antes
de actuar, **revisión humana**: nunca un castigo automático.

## Ahora tú

Modifica y **observa cómo cambian las decisiones** (las soluciones se publican después de la sesión):

1. **Elegir k.** Vuelve a segmentar con `n_clusters = 2` y con `5`, y compara los promedios por
   grupo. ¿Cuántos segmentos son realmente **accionables** para el negocio? Justifica tu elección.
2. **Nombrar y decidir.** Para el k que elegiste, escribe una tabla con **nombre** y una **acción**
   por segmento. Si solo pudieras hacer **una** campaña, ¿a qué grupo se la darías y por qué?
3. **Anomalías.** Cambia `fraction` a `0.02` y a `0.10` en el Isolation Forest. ¿Cuántos clientes se
   marcan en cada caso? ¿Qué implica marcar **más** o **menos** para el equipo que debe revisarlos?
4. **PCA — el otro eje.** Mira los pesos del **Componente 2** (`pca.components_[1]`, con
   `cancer.data.columns`). ¿Qué variables pesan más? ¿Cómo **lo nombrarías**?

In [18]:
# 1) Tu código aquí: prueba n_clusters = 2 y 5, y compara los promedios por grupo.

In [19]:
# 3) Tu código aquí: cambia fraction (0.02 y 0.10) y cuenta cuántos clientes quedan marcados.

---

### Para cerrar

- **Segmentamos** sin etiqueta y **perfilamos** los grupos para accionar.
- **PCA** resumió 30 variables en pocos ejes para *ver* la estructura (sin predecir).
- **Isolation Forest** marcó lo que se sale del patrón.

En no supervisado la "verdad" no está en los datos: está en la **utilidad** de lo que descubres.

*Las soluciones de los ejercicios se publican después de la sesión.*

<!-- migracion-modular: atribucion-datos -->
## Atribución de datos

**Creador:** Equipo docente del módulo.
**Procedencia:** Datos generados con [`generar_clientes.py`](../../../scripts/generar_clientes.py), con semilla fija.
**Nota:** Datos sintéticos; no representan a una organización real ni a personas reales.
**Modificación:** se omite la columna `fuga` durante la segmentación y se conservan columnas didácticas.
